# Load-flow infeasibility diagnosis

This notebook finds failed load-flow hours, reads the solver's IIS when available, and produces an interactive Folium map of the implicated buses and network elements. It includes both transmission lines and transformers.

For a new run, set `RESULTS_DIR` to its results folder and run all cells. The solver now writes hour-specific IIS files (`iis_YYYY-MM-DD_hHH.txt`) for DC infeasibilities; the notebook also supports the legacy `iis.txt` file. For AC/IPOPT failures without an IIS, it maps the most heavily loaded elements in the nearest solved hour as evidence, but labels that result as a heuristic.

In [6]:
from pathlib import Path
import re
import html
import webbrowser

import numpy as np
import pandas as pd
import folium

PROJECT = Path.cwd()

# Base 2024 run: PROJECT / 'results'.  Example scenario: PROJECT / 'results' / 'GoRES'.
RESULTS_DIR = PROJECT / 'results' / 'GoRES'

# Leave both as None to diagnose the first failed hour automatically.
# Example: DIAG_DATE, DIAG_HOUR = '2024-12-02', 5
DIAG_DATE = None
DIAG_HOUR = None

# The AC (Ipopt) path reports success as LOCALLY_SOLVED and the DC (Gurobi) path as
# OPTIMAL; anything else is a failed hour.  Listing only OPTIMAL here would mark every
# AC hour as failed and silently diagnose an hour that actually solved.
SOLVED_STATUSES = {'OPTIMAL', 'LOCALLY_SOLVED', 'ALMOST_OPTIMAL', 'ALMOST_LOCALLY_SOLVED'}

# iis.txt (no hour suffix) is a copy of the most recent *DC* diagnosis and may belong to
# a different day, hour, or run entirely.  Leave this False so an AC failure is not
# explained with a stale conflict; set it True only when iis.txt is known to be current.
ALLOW_LEGACY_IIS = False

# PowerModels network base used by this project.  IIS pmin/rate values are in pu.
BASE_MVA = 100.0
OPEN_BROWSER = True  # open the saved Folium HTML in the system/default browser
OUT = RESULTS_DIR / 'grid_maps'
OUT.mkdir(parents=True, exist_ok=True)
print(f'Results directory: {RESULTS_DIR}')

Results directory: c:\Users\ehsanno\DataspellProjects\Spanish_Power_System\results\GoRES


In [7]:
# Shared topology.  Unlike grid_plotting.ipynb, transformers are included because
# transformer bottlenecks can be the direct cause of an infeasibility.
buses = pd.read_csv(PROJECT / 'Data' / 'Bus_Data.csv', encoding='utf-8-sig')
BUS = buses.set_index('bus_id')
BUS_BY_I = dict(enumerate(buses['bus_id'], start=1))

lines = pd.read_csv(PROJECT / 'Data' / 'lines.csv')
lines = lines[lines['bus0'].isin(BUS.index) & lines['bus1'].isin(BUS.index)].copy()
lines['element_id'] = lines['line_id'].astype(str)
lines['kind'] = np.where(lines['dc'].astype(str).str.lower().isin(['t', 'true']), 'HVDC line', 'AC line')
lines['nameplate_mw'] = (np.sqrt(3) * lines['voltage'] * lines['Imax'] * lines['circuits'].clip(lower=1))

transformers = pd.read_csv(PROJECT / 'Data' / 'transformers_reactance.csv')
transformers = transformers[transformers['bus0'].isin(BUS.index) & transformers['bus1'].isin(BUS.index)].copy()
transformers['element_id'] = transformers['transformer_id'].astype(str)
transformers['kind'] = 'transformer'
transformers['nameplate_mw'] = transformers['installed_MVA']

TOPO = pd.concat([
    lines[['element_id', 'bus0', 'bus1', 'kind', 'nameplate_mw', 'voltage']],
    transformers[['element_id', 'bus0', 'bus1', 'kind', 'nameplate_mw', 'HV_kV']],
], ignore_index=True).rename(columns={'HV_kV': 'voltage'})
TOPO = TOPO.drop_duplicates('element_id').set_index('element_id', drop=False)

summary_file = RESULTS_DIR / 'summary.csv'
if not summary_file.exists():
    raise FileNotFoundError(f'No summary.csv: {summary_file}')
summary = pd.read_csv(summary_file)
failed = summary[~summary['status'].isin(SOLVED_STATUSES)].copy()
if failed.empty:
    raise RuntimeError('summary.csv contains no non-optimal load-flow hours.')

if DIAG_DATE is None or DIAG_HOUR is None:
    target = failed.iloc[0]
else:
    matching = failed[(failed['date'].astype(str) == str(DIAG_DATE)) & (failed['hour'] == DIAG_HOUR)]
    if matching.empty:
        raise ValueError(f'{DIAG_DATE} h{DIAG_HOUR:02d} is not a failed hour. Available:\n{failed[["date", "hour", "status"]].to_string(index=False)}')
    target = matching.iloc[0]

target_date, target_hour = str(target['date']), int(target['hour'])
print('Failed load-flow hours:')
display(failed[['date', 'hour', 'status', 'total_load_mw']])
print(f'\nSelected: {target_date} h{target_hour:02d} ({target.status})')

Failed load-flow hours:


,date,hour,status,total_load_mw
0,2024-07-08,0,INFEASIBLE,39174.245788
1,2024-07-08,1,INFEASIBLE,37177.817959
3,2024-07-08,3,INFEASIBLE,34455.415875
4,2024-07-08,4,INFEASIBLE,33901.968425
5,2024-07-08,5,INFEASIBLE,34513.737978
6,2024-07-08,6,INFEASIBLE,38432.073541
7,2024-07-08,7,INFEASIBLE,42169.093670
12,2024-07-08,12,INFEASIBLE,61479.866442
22,2024-07-08,22,LOCALLY_SOLVED,49010.400000
23,2024-07-08,23,INFEASIBLE,46857.045633



Selected: 2024-07-08 h00 (INFEASIBLE)


In [8]:
def iis_for_hour(results_dir, date, hour):
    hourly = results_dir / f'iis_{date}_h{hour:02d}.txt'
    legacy = results_dir / 'iis.txt'
    if hourly.exists():
        return hourly, False
    if legacy.exists() and ALLOW_LEGACY_IIS:
        return legacy, True
    return None, False

def parse_iis(path):
    """Extract IIS member generators, branches, and buses plus the readable legend."""
    raw = path.read_text(encoding='utf-8', errors='replace')
    # The legend separator has box-drawing dashes; split at its *first* occurrence.
    # A greedy regex here would incorrectly discard all but the first constraint.
    conflict = raw.split('legend', 1)[0]
    # The conflict is deliberately preserved: it is the solver's exact evidence.
    # JuMP prints variables as 0_pg[...] / 0_p[...]; '_' is a regex word character,
    # so do not require a word boundary before the variable name.
    pg_ids = sorted({int(x) for x in re.findall(r'pg\[(\d+)\]', conflict)})
    p_refs = [(int(a), int(b), int(c)) for a, b, c in
              re.findall(r'p\[\((\d+),\s*(\d+),\s*(\d+)\)\]', conflict)]
    branch_ids = sorted({a for a, _, _ in p_refs})

    gens, branches, bus_names = {}, {}, {}
    for idx, name, bus_i, fuel, pmin, pmax in re.findall(
        r'pg\[(\d+)\] = (.+?) @ bus (\d+) \((.+?), (.+?)\.\.(.+?)\)', raw):
        gens[int(idx)] = dict(name=name, bus_i=int(bus_i), fuel=fuel, pmin=float(pmin), pmax=float(pmax))
    for idx, name, fb, tb, rate, x in re.findall(
        r'br\[(\d+)\] = (.+?) (\d+)→(\d+) \(rate ([^,]+), x ([^)]+)\)', raw):
        branches[int(idx)] = dict(name=name, from_i=int(fb), to_i=int(tb), rate=float(rate), x=float(x))
    for idx, name in re.findall(r'bus (\d+) = ([^\r\n]+)', raw):
        bus_names[int(idx)] = name.strip()

    used_buses = {b for _, b, c in p_refs for b in (b, c)}
    used_buses |= {gens[g]['bus_i'] for g in pg_ids if g in gens}
    return dict(raw=raw, conflict=conflict, pg_ids=pg_ids, branch_ids=branch_ids,
                p_refs=p_refs, gens=gens, branches=branches, bus_names=bus_names,
                bus_ids=used_buses)

iis_path, legacy_iis = iis_for_hour(RESULTS_DIR, target_date, target_hour)
iis = parse_iis(iis_path) if iis_path else None
if iis_path:
    note = 'legacy file — verify it belongs to the selected hour' if legacy_iis else 'hour-specific IIS'
    print(f'IIS source: {iis_path.name} ({note})')
    print('\nExact conflicting constraints from the solver:\n' + iis['conflict'].strip())
else:
    print('No IIS was found. The map will use the nearest solved hour as a congestion heuristic.')

IIS source: iis.txt (legacy file — verify it belongs to the selected hour)

Exact conflicting constraints from the solver:
-0_pg[904] - 0_pg[2352] - 0_pg[899] - 0_pg[874] - 0_pg[1737] - 0_pg[881] - 0_pg[911] - 0_pg[875] - 0_p[(1523, 1027, 1028)] - 0_p[(1522, 1027, 1028)] == -0.05383063606303942
-0_pg[1139] - 0_p[(1857, 1055, 970)] == 0
-0_pg[2435] - 0_p[(1521, 1027, 1054)] + 0_p[(2327, 1054, 1055)] == -0.011304341276073797
-0_pg[1736] - 0_pg[2702] + 0_p[(1521, 1027, 1054)] + 0_p[(1523, 1027, 1028)] - 0_p[(1520, 212, 1027)] + 0_p[(1522, 1027, 1028)] == -0.009855176776598536
-0_p[(2327, 1054, 1055)] + 0_p[(1857, 1055, 970)] == 0
0_pg[1736] >= -2.1982751084739743e-5
0_pg[904] >= 0.12369999999999999
0_pg[2352] >= 0
0_pg[899] >= 0.19829999999999998
0_pg[874] >= 0.32020000000000004
0_pg[1737] >= -1.465516738982649e-5
0_pg[1139] >= 2.165513833992095
0_pg[2435] >= 0
0_pg[881] >= 1.4997
0_pg[911] >= 0.0055000000000000005
0_pg[875] >= 0.5457
0_pg[2702] >= 0
0_p[(1520, 212, 1027)] >= -3.932448153

In [9]:
flows_file = RESULTS_DIR / 'branch_flows.csv'
flows = pd.read_csv(flows_file) if flows_file.exists() and flows_file.stat().st_size else pd.DataFrame()

def nearest_solved_flow(date, hour, names=None):
    if flows.empty:
        return pd.DataFrame(), None
    data = flows.copy()
    if names is not None:
        data = data[data['branch_name'].isin(names)]
    # Prefer same-day adjacent feasible hours; then any closest saved hour.
    data['_distance'] = (data['date'].astype(str).ne(str(date)).astype(int) * 1000
                         + (data['hour'] - hour).abs())
    if data.empty:
        return data, None
    ref = data.sort_values(['_distance', 'date', 'hour']).iloc[0]
    rows = data[(data['date'] == ref['date']) & (data['hour'] == ref['hour'])].drop(columns='_distance')
    return rows, (str(ref['date']), int(ref['hour']))

if iis:
    conflict_branches = [iis['branches'][i]['name'] for i in iis['branch_ids'] if i in iis['branches']]
    conflict_buses_i = iis['bus_ids']
    conflict_buses = {iis['bus_names'].get(i, BUS_BY_I.get(i, f'bus {i}')) for i in conflict_buses_i}
    evidence, reference_hour = nearest_solved_flow(target_date, target_hour, conflict_branches)
    branch_rows = []
    for idx in iis['branch_ids']:
        if idx not in iis['branches']:
            continue
        b = iis['branches'][idx]
        element = TOPO.loc[b['name']] if b['name'] in TOPO.index else None
        f = evidence[evidence['branch_name'].eq(b['name'])]
        branch_rows.append({
            'branch_id': idx, 'element': b['name'],
            'type': element['kind'] if element is not None else 'not in mappable topology',
            'from_bus': iis['bus_names'].get(b['from_i'], BUS_BY_I.get(b['from_i'])),
            'to_bus': iis['bus_names'].get(b['to_i'], BUS_BY_I.get(b['to_i'])),
            'IIS limit MW': b['rate'] * BASE_MVA,
            'nearest flow MW': None if f.empty else float(f.iloc[0]['flow_mw']),
            'nearest loading %': None if f.empty else float(f.iloc[0]['loading_pct']),
        })
    generator_rows = [{
        'gen_id': idx, 'unit': iis['gens'][idx]['name'],
        'bus': iis['bus_names'].get(iis['gens'][idx]['bus_i'], BUS_BY_I.get(iis['gens'][idx]['bus_i'])),
        'fuel': iis['gens'][idx]['fuel'],
        'Pmin MW': iis['gens'][idx]['pmin'] * BASE_MVA,
        'Pmax MW': iis['gens'][idx]['pmax'] * BASE_MVA,
    } for idx in iis['pg_ids'] if idx in iis['gens']]
    elements_to_map = set(conflict_branches)
    diagnosis_mode = 'IIS (exact conflict)'
else:
    evidence, reference_hour = nearest_solved_flow(target_date, target_hour)
    evidence = evidence.sort_values('loading_pct', ascending=False).head(12)
    branch_rows = evidence.rename(columns={'branch_name': 'element', 'limit_mw': 'IIS limit MW',
                                            'flow_mw': 'nearest flow MW', 'loading_pct': 'nearest loading %'})
    branch_rows = branch_rows[['element', 'from_bus', 'to_bus', 'IIS limit MW', 'nearest flow MW', 'nearest loading %']].copy()
    branch_rows['type'] = branch_rows['element'].map(TOPO['kind']).fillna('unknown')
    generator_rows = []
    elements_to_map = set(branch_rows['element'])
    conflict_buses = set()
    for name in elements_to_map:
        if name in TOPO.index:
            conflict_buses.update([TOPO.at[name, 'bus0'], TOPO.at[name, 'bus1']])
    diagnosis_mode = 'nearest-solved-hour heuristic'

print(f'Diagnosis mode: {diagnosis_mode}')
if reference_hour:
    print(f'Flow evidence is from {reference_hour[0]} h{reference_hour[1]:02d}; the selected failed hour has no feasible flow solution.')
print('\nNetwork elements:')
display(pd.DataFrame(branch_rows))
if generator_rows:
    print('\nFixed/limited generators in the IIS:')
    display(pd.DataFrame(generator_rows))

Diagnosis mode: IIS (exact conflict)
Flow evidence is from 2024-07-08 h02; the selected failed hour has no feasible flow solution.

Network elements:


,branch_id,element,type,from_bus,to_bus,IIS limit MW,nearest flow MW,nearest loading %
0,1520,LTGES1001,AC line,ES00215,ES01041,393.0,-384.97,97.9
1,1521,LTGES1002,AC line,ES01041,ES01068,393.0,-120.66,30.7
2,1522,LTGES1003a,AC line,ES01041,ES01042,786.0,-132.54,16.9
3,1523,LTGES1003b,AC line,ES01041,ES01042,786.0,-132.54,16.9
4,1857,LTGES1219,AC line,ES01069,ES00981,402.0,-121.55,30.2
5,2327,TR_fa1d5cde,transformer,ES01068,ES01069,384.0,-121.55,31.7



Fixed/limited generators in the IIS:


,gen_id,unit,bus,fuel,Pmin MW,Pmax MW
0,874,G0929,ES01042,Hydro,32.0,32.0
1,875,G0930,ES01042,Hydro,54.6,54.6
2,881,G0936,ES01042,Hydro,150.0,150.0
3,899,G0954,ES01042,Hydro,19.8,19.8
4,904,G0959,ES01042,Hydro,12.4,12.4
5,911,G0966,ES01042,Hydro,0.6,0.6
6,1139,XB_ES00981,ES00981,CrossBorder,216.6,216.6
7,1736,BESS_ES01041,ES01041,Battery,-0.0,0.0
8,1737,BESS_ES01042,ES01042,Battery,-0.0,0.0
9,2352,LS_ES01042,ES01042,LoadShed,0.0,5.4


In [10]:
def flow_lookup(rows):
    if rows.empty:
        return {}
    return {r.branch_name: r for r in rows.itertuples()}

def add_legend(m, title, items):
    rows = '<br>'.join(f'<span style="display:inline-block;width:18px;height:4px;background:{color};margin-right:6px;vertical-align:middle"></span>{label}' for color, label in items)
    folium.Element(f'<div style="position:fixed;bottom:28px;left:18px;z-index:9999;background:white;padding:8px 11px;border:1px solid #888;border-radius:5px;font:12px/1.5 sans-serif"><b>{title}</b><br>{rows}</div>').add_to(m.get_root().html)

def plot_infeasibility():
    m = folium.Map(location=[40.0, -3.6], zoom_start=6, tiles='CartoDB positron', control_scale=True)
    fg_base = folium.FeatureGroup(name='network', show=True)
    fg_issue = folium.FeatureGroup(name='IIS / likely constraints', show=True)
    fg_bus = folium.FeatureGroup(name='implicated buses', show=True)
    f_by_name = flow_lookup(evidence)

    for r in TOPO.itertuples():
        b0, b1 = BUS.loc[r.bus0], BUS.loc[r.bus1]
        pts = [(b0.y, b0.x), (b1.y, b1.x)]
        is_issue = r.element_id in elements_to_map
        style = dict(color='#d73027', weight=5, opacity=0.95) if is_issue else dict(color='#9aa0a6', weight=1, opacity=0.5)
        if r.kind == 'transformer' and not is_issue:
            style['dash_array'] = '4,5'
        flow = f_by_name.get(r.element_id)
        flow_text = '' if flow is None else f'<br>nearest solved flow: {flow.flow_mw:.1f} MW ({flow.loading_pct:.1f}% of limit)'
        rating = '' if pd.isna(r.nameplate_mw) else f'<br>nameplate: {r.nameplate_mw:.1f} MW'
        tip = f'<b>{html.escape(r.element_id)}</b><br>{r.kind}{rating}{flow_text}'
        folium.PolyLine(pts, tooltip=folium.Tooltip(tip, sticky=True), **style).add_to(fg_issue if is_issue else fg_base)

    gen_at_bus = {}
    for row in generator_rows:
        gen_at_bus.setdefault(row['bus'], []).append(row)
    for bus_id in sorted(conflict_buses):
        if bus_id not in BUS.index:
            continue
        b = BUS.loc[bus_id]
        entries = gen_at_bus.get(bus_id, [])
        gen_text = ''.join(f'<br>{html.escape(g["unit"])}: Pmin {g["Pmin MW"]:.2f} MW' for g in entries)
        folium.CircleMarker([b.y, b.x], radius=7, color='#7a001f', weight=2, fill=True,
                            fill_color='#ffcc00', fill_opacity=0.95,
                            tooltip=folium.Tooltip(f'<b>{bus_id}</b> | {int(b.voltage)} kV{gen_text}', sticky=True)).add_to(fg_bus)

    for layer in (fg_base, fg_issue, fg_bus):
        layer.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    add_legend(m, 'Infeasibility map', [('#d73027', 'IIS / likely constraining element'), ('#ffcc00', 'IIS / connected bus'), ('#9aa0a6', 'other network element')])

    mapped_buses = [b for b in conflict_buses if b in BUS.index]
    if mapped_buses:
        points = [(BUS.loc[b, 'y'], BUS.loc[b, 'x']) for b in mapped_buses]
        m.fit_bounds([[min(x[0] for x in points), min(x[1] for x in points)], [max(x[0] for x in points), max(x[1] for x in points)]], padding=(50, 50))
    return m

m = plot_infeasibility()
out = OUT / f'infeasibility_{target_date}_h{target_hour:02d}.html'
m.save(out)
print(f'Saved interactive map: {out}')
if OPEN_BROWSER:
    webbrowser.open(out.resolve().as_uri())
m

Saved interactive map: c:\Users\ehsanno\DataspellProjects\Spanish_Power_System\results\GoRES\grid_maps\infeasibility_2024-07-08_h00.html


## How to interpret the result

- Red elements and yellow buses are the exact members of the IIS when an IIS is available. Together, their constraints cannot all be satisfied.
- `Pmin MW` identifies a fixed or minimum injection. Compare it with the IIS element limit and the nearby-hour loading. A fixed injection that slightly exceeds the only outgoing transformer/line is a direct infeasibility mechanism.
- When no IIS exists (common for an AC/IPOPT local infeasibility), the red elements are only the highest-loaded elements in a nearby solved hour. Use them as a starting point, not as a proof.
- Select another failed hour in the settings cell. New DC diagnosis runs retain a separate IIS file per hour, so multiple failures can be mapped independently.